# 第4章 高级子查询

## 📌 学习目标

- ALL / ANY / SOME 子查询
- 派生表（Derived Tables）
- SELECT 中的标量子查询
- 子查询与 JOIN 的性能对比

---

In [ ]:
import os
import pymysql
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": "shop_db",
    "charset": "utf8mb4",
}

def run_query(sql, params=None):
    conn = pymysql.connect(**DB_CONFIG)
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

## 1. ALL 子查询

`> ALL(子查询)` 表示大于子查询返回的**所有值**（即大于最大值）。

```sql
WHERE 列 > ALL(SELECT ...)  -- 大于所有值
WHERE 列 < ALL(SELECT ...)  -- 小于所有值
```

In [ ]:
# 查询价格高于"食品"分类所有商品的商品
run_query("""
    SELECT product_name, price
    FROM products
    WHERE price > ALL (
        SELECT price FROM products WHERE category_id = 3
    )
    ORDER BY price
""")

In [ ]:
# 等价写法：使用 MAX
run_query("""
    SELECT product_name, price
    FROM products
    WHERE price > (SELECT MAX(price) FROM products WHERE category_id = 3)
    ORDER BY price
""")

## 2. ANY / SOME 子查询

`> ANY(子查询)` 表示大于子查询返回的**任意一个值**（即大于最小值）。

`ANY` 和 `SOME` 是同义词。

In [ ]:
# 查询价格高于"食品"分类中任一商品的商品
run_query("""
    SELECT product_name, price
    FROM products
    WHERE price > ANY (
        SELECT price FROM products WHERE category_id = 3
    )
    ORDER BY price
""")

In [ ]:
# 等价写法：使用 MIN
run_query("""
    SELECT product_name, price
    FROM products
    WHERE price > (SELECT MIN(price) FROM products WHERE category_id = 3)
    ORDER BY price
""")

## 3. SELECT 中的标量子查询

在 SELECT 列表中嵌入子查询，为每一行计算一个值。

In [ ]:
# 每个商品与平均价格的对比
run_query("""
    SELECT 
        product_name,
        price,
        (SELECT ROUND(AVG(price), 2) FROM products) AS 平均价格,
        price - (SELECT AVG(price) FROM products) AS 与均价差异
    FROM products
    ORDER BY price DESC
""")

In [ ]:
# 每个客户的订单数和总消费（用标量子查询）
run_query("""
    SELECT 
        customer_name,
        city,
        (SELECT COUNT(*) FROM orders WHERE customer_id = c.customer_id) AS 订单数,
        (SELECT IFNULL(SUM(total_amount), 0) FROM orders WHERE customer_id = c.customer_id) AS 总消费
    FROM customers c
    ORDER BY 总消费 DESC
""")

## 4. 派生表（Derived Tables）

在 FROM 子句中的子查询，也叫内联视图（Inline View）。

In [ ]:
# 派生表：先聚合再连接
run_query("""
    SELECT 
        c.customer_name,
        cs.order_count,
        cs.total_spent
    FROM customers c
    INNER JOIN (
        SELECT 
            customer_id,
            COUNT(*) AS order_count,
            SUM(total_amount) AS total_spent
        FROM orders
        GROUP BY customer_id
    ) cs ON c.customer_id = cs.customer_id
    ORDER BY cs.total_spent DESC
""")

In [ ]:
# 多层派生表
run_query("""
    SELECT *
    FROM (
        SELECT 
            category_id,
            COUNT(*) AS product_count,
            AVG(price) AS avg_price
        FROM products
        GROUP BY category_id
    ) cat_stats
    WHERE product_count > 3
    ORDER BY avg_price DESC
""")

## 📝 练习题

### 题目1（简单）
使用 ALL 查询工资高于"技术部"所有员工的员工。

### 题目2（中等）
使用 SELECT 中的标量子查询，显示每个商品的名称、价格、所属分类的平均价格。

### 题目3（中等）
使用派生表查询每个城市消费最高的客户。

---

## ✅ 参考答案

In [ ]:
# 题目1：ALL 查询
run_query("""
    SELECT employee_name, department, salary
    FROM employees
    WHERE salary > ALL (
        SELECT salary FROM employees WHERE department = '技术部'
    )
    ORDER BY salary DESC
""")

In [ ]:
# 题目2：标量子查询
run_query("""
    SELECT 
        p.product_name,
        p.price,
        (SELECT ROUND(AVG(price), 2) FROM products WHERE category_id = p.category_id) AS 分类均价
    FROM products p
    ORDER BY p.category_id, p.price DESC
""")

In [ ]:
# 题目3：派生表查询每城市消费最高客户
run_query("""
    SELECT c.customer_name, c.city, cs.total_spent
    FROM customers c
    INNER JOIN (
        SELECT customer_id, SUM(total_amount) AS total_spent
        FROM orders
        GROUP BY customer_id
    ) cs ON c.customer_id = cs.customer_id
    WHERE cs.total_spent = (
        SELECT MAX(sub.total_spent)
        FROM (
            SELECT o2.customer_id, SUM(o2.total_amount) AS total_spent
            FROM orders o2
            INNER JOIN customers c2 ON o2.customer_id = c2.customer_id
            WHERE c2.city = c.city
            GROUP BY o2.customer_id
        ) sub
    )
    ORDER BY c.city
""")